# Exercise 7, the lumped hydrological model

These exercises go with **Lecture 7 &mdash; A lumped hydrological model**. Keep that
notebook open beside this one: you will reuse its `Catchment` class and the Chattooga
data throughout.

Each exercise has the same shape:

* a short **background** recap so the exercise is self-contained,
* a **task** broken into numbered steps,
* **hints**, and
* code cells that are already wired up &mdash; the data loading, the model, and every
  plot are written for you. You only fill in the cells marked

  ```python
  # ==== YOUR CODE ====
  ```

  following the comments inside them. Cells below your code will not run correctly
  until you have filled it in.


In [ ]:
# Google Colab setup: installs the packages this notebook uses (runs only on Colab).
import sys
if "google.colab" in sys.modules:
    %pip install -q numpy pandas matplotlib scipy xarray netcdf4 pooch


## Setup (given &mdash; just run it)

This cell reproduces the model and data from Lecture 7. Run it and move on.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


class Catchment:
    """The lumped daily rainfall-runoff model from Lecture 7.

    One soil-moisture store feeds a fast pathway (two linear reservoirs in series)
    and a slow pathway (one linear reservoir).
    """

    def __init__(self, smax=250.0, beta=2.5, kperc=0.05, kf=0.4, ks=0.015):
        self.smax = smax; self.beta = beta; self.kperc = kperc
        self.kf = kf; self.ks = ks

    def run(self, prcp, pet, s0=0.5, return_stores=False):
        smax, beta, kperc = self.smax, self.beta, self.kperc
        kf, ks = self.kf, self.ks
        prcp = np.asarray(prcp, float); pet = np.asarray(pet, float)
        S = s0 * smax
        F1 = F2 = R = 0.0
        n = len(prcp); q = np.empty(n); total_store = np.empty(n)
        for t in range(n):
            p, ep = prcp[t], pet[t]
            s = min(max(S / smax, 0.0), 1.0)
            runoff = p * s ** beta
            aet = ep * s * (2.0 - s)
            S = S + p - runoff - aet
            if S > smax:
                runoff += S - smax; S = smax
            if S < 0.0:
                aet += S; S = 0.0
            perc = kperc * S * s ** 3
            S -= perc
            F1 += runoff; R += perc
            o1 = kf * F1; F1 -= o1; F2 += o1
            o2 = kf * F2; F2 -= o2
            qs = ks * R; R -= qs
            q[t] = o2 + qs
            total_store[t] = S + F1 + F2 + R
        return (q, total_store) if return_stores else q

    def run_split(self, prcp, pet, s0=0.5):
        """Like run(), but returns (q_fast, q_slow) as separate arrays."""
        smax, beta, kperc = self.smax, self.beta, self.kperc
        kf, ks = self.kf, self.ks
        prcp = np.asarray(prcp, float); pet = np.asarray(pet, float)
        S = s0 * smax; F1 = F2 = R = 0.0
        n = len(prcp); qf = np.empty(n); qs = np.empty(n)
        for t in range(n):
            p, ep = prcp[t], pet[t]
            s = min(max(S / smax, 0.0), 1.0)
            runoff = p * s ** beta
            aet = ep * s * (2.0 - s)
            S = S + p - runoff - aet
            if S > smax:
                runoff += S - smax; S = smax
            if S < 0.0:
                aet += S; S = 0.0
            perc = kperc * S * s ** 3; S -= perc
            F1 += runoff; R += perc
            o1 = kf * F1; F1 -= o1; F2 += o1
            o2 = kf * F2; F2 -= o2
            slow = ks * R; R -= slow
            qf[t] = o2; qs[t] = slow
        return qf, qs


def nse(obs, sim):
    """Nash-Sutcliffe efficiency (1 = perfect, 0 = no better than the mean)."""
    m = np.isfinite(obs) & np.isfinite(sim)
    o, s = obs[m], sim[m]
    return 1.0 - np.sum((s - o) ** 2) / np.sum((o - o.mean()) ** 2)


df = pd.read_csv("data/chattooga_daily.csv", comment="#",
                 parse_dates=["date"], index_col="date")
P   = df["prcp_mm"].to_numpy()
Ep  = df["pet_mm"].to_numpy()
T   = df["tmean_c"].to_numpy()
Qobs = df["q_mm"].to_numpy()

# the hand-picked parameter set from Lecture 7
BASE = dict(smax=250.0, beta=2.5, kperc=0.05, kf=0.40, ks=0.015)
EVAL = df.index >= "1993-01-01"          # skip the 2-year warm-up when scoring

q_base = Catchment(**BASE).run(P, Ep)
print(f"baseline model (Lecture 7 parameters):  NSE = {nse(Qobs[EVAL], q_base[EVAL]):.3f}")


## Exercise 1 &mdash; add a snow store

The Chattooga is a warm catchment, so Lecture 7 could feed rainfall straight into the
soil. In a Himalayan or high-latitude catchment a large part of the winter precipitation
is *stored as snow* and released weeks or months later as melt. This exercise adds that
store.

**Background.** A **degree-day** snow model is the simplest one that works. Each day,
with air temperature $T$ (°C), precipitation $P$ (mm), a snowpack $M$ (mm water
equivalent) and a degree-day factor $f$ (mm °C⁻¹ day⁻¹):

* if $T \le 0$: all precipitation is snow &mdash; $M \leftarrow M + P$, and the liquid
  water reaching the soil that day is $0$;
* if $T > 0$: no new snow; the melt is $m = \min(M,\; f\,T)$, so
  $M \leftarrow M - m$, and the liquid water reaching the soil is $P + m$.

You then run the **same** `Catchment` model, but forced with this *liquid input* series
instead of raw precipitation.

**Your task.**
1. Complete the function `snow_module(P, T, ddf)` in the cell below.
2. Build the liquid-input series with `ddf = 3.0` and run the catchment model on it.
3. Compare the snow-corrected hydrograph with the baseline (the given plot does this).

**Hints.**
* Loop over days with a running variable `M` (start at `0.0`), exactly like the loop
  inside `Catchment.run`.
* `np.minimum` is not needed &mdash; plain `min(M, ddf * T[t])` works on scalars.
* The Chattooga rarely drops below 0 °C, so the effect here is small. To *see* it,
  the plot also shows a synthetic cold version of the same catchment (`T - 12`).


In [ ]:
def snow_module(P, T, ddf=3.0):
    """Degree-day snow store.

    Parameters
    ----------
    P, T : arrays of daily precipitation [mm] and mean air temperature [degC]
    ddf  : degree-day melt factor [mm/degC/day]

    Returns
    -------
    liquid_input : array, same length as P, of rain + snowmelt [mm/day]
    """
    n = len(P)
    liquid_input = np.empty(n)
    M = 0.0                              # snow water equivalent on the ground [mm]

    for t in range(n):
        # ==== YOUR CODE ====
        # Step 1: is it freezing?  (T[t] <= 0)
        #    - if yes:  add P[t] to M;  today's liquid input is 0.0
        #    - if no :  melt = min(M, ddf * T[t]);  subtract melt from M;
        #               today's liquid input is P[t] + melt
        # Step 2: store the result:  liquid_input[t] = ...
        liquid_input[t] = 0.0           # <-- replace this line
        # ===================

    return liquid_input


# --- build the liquid-input series and run the model (given, once snow_module works) ---
liquid = snow_module(P, T, ddf=3.0)
q_snow = Catchment(**BASE).run(liquid, Ep)

# a synthetic COLD version of the catchment, to make the snow effect visible
T_cold = T - 12.0
liquid_cold = snow_module(P, T_cold, ddf=3.0)
q_cold_nosnow = Catchment(**BASE).run(P,        Ep)     # cold climate, snow ignored
q_cold_snow   = Catchment(**BASE).run(liquid_cold, Ep)  # cold climate, snow modelled

print(f"Chattooga as-is        : NSE {nse(Qobs[EVAL], q_snow[EVAL]):.3f} "
      f"(baseline {nse(Qobs[EVAL], q_base[EVAL]):.3f})")
print(f"cold-climate run: {100 * np.mean(T_cold < 0):.0f}% of days below freezing; "
      f"of the {P.sum():.0f} mm total precip, "
      f"{np.maximum(P - liquid_cold, 0).sum():.0f} mm passed through the snowpack")


In [ ]:
# --- plot (given) ---
t = df.index
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

sl = slice("2009", "2011")
ax[0].plot(df.loc[sl].index, pd.Series(Qobs, index=t).loc[sl], color="black", lw=0.9, label="observed")
ax[0].plot(df.loc[sl].index, pd.Series(q_base, index=t).loc[sl], color="tab:blue", lw=0.9, label="baseline (no snow)")
ax[0].plot(df.loc[sl].index, pd.Series(q_snow, index=t).loc[sl], color="tab:red", lw=0.9, ls="--", label="with snow module")
ax[0].set_ylabel("Q [mm/d]"); ax[0].set_title("Chattooga (mild climate): snow module barely matters")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

ax[1].plot(df.loc[sl].index, pd.Series(q_cold_nosnow, index=t).loc[sl], color="tab:blue", lw=0.9, label="cold climate, snow ignored")
ax[1].plot(df.loc[sl].index, pd.Series(q_cold_snow, index=t).loc[sl], color="tab:red", lw=0.9, label="cold climate, snow modelled")
ax[1].set_ylabel("Q [mm/d]"); ax[1].set_xlabel("year")
ax[1].set_title("same catchment shifted 12 degC colder: snow delays winter runoff into a spring melt peak")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
Write your answers here.

1. In the cold-climate run, when does runoff now peak, and why is that different from
   the "snow ignored" run?
2. The degree-day factor `ddf` is the one new parameter. Try `ddf = 1.5` and `ddf = 6`.
   What does a larger `ddf` do to the shape and timing of the melt peak?
3. This is the catchment-scale version of which feedback from Lectures 1 and 5?
```


## Exercise 2 &mdash; calibrate to a signature, not the hydrograph

Lecture 8 fits the model to the whole hydrograph. Often you want the model to get one
*hydrological signature* right &mdash; a single number that summarises some aspect of
the flow regime. Here you tune **one parameter** to match the most common signature,
the **baseflow index**.

**Background.** The baseflow index (BFI) is the fraction of total streamflow that comes
from slow storage rather than from storm runoff:

$$\text{BFI} = \frac{\sum \text{baseflow}}{\sum \text{streamflow}}.$$

For the model this is easy &mdash; `Catchment.run_split` returns the fast and slow
components separately, so $\text{BFI}_\text{model} = \sum q_\text{slow} / \sum q_\text{total}$.
For the *observed* series there is no such split, so hydrologists estimate baseflow with
a recursive digital filter. The **Lyne&ndash;Hollick** filter (given below, complete)
is the standard one. In the model, the parameter that controls how much water takes the
slow path is $k_{perc}$ (percolation to groundwater), so that is the knob you turn.

**Your task.**
1. Use the given `lyne_hollick` filter to get the observed baseflow, then compute
   $\text{BFI}_\text{obs}$ (one line).
2. Loop $k_{perc}$ over `kperc_grid`; for each value run `run_split` and record the
   model BFI.
3. Pick the $k_{perc}$ whose model BFI is closest to $\text{BFI}_\text{obs}$.

**Hints.**
* `qf, qs = Catchment(**pars).run_split(P, Ep)` then
  `bfi = qs[EVAL].sum() / (qf[EVAL] + qs[EVAL]).sum()`.
* `np.argmin(np.abs(bfi_array - BFI_obs))` gives the index of the closest match.
* Score only over `EVAL` (post warm-up), as everywhere else.


In [ ]:
# --- Lyne-Hollick recursive digital baseflow filter (given, complete) ---
def lyne_hollick(q, alpha=0.925, passes=3):
    """Return the baseflow series for streamflow `q` (no NaNs)."""
    q = np.asarray(q, float)
    b = q.copy()
    for p in range(passes):
        qf = np.zeros_like(b)
        rng = range(1, len(b)) if p % 2 == 0 else range(len(b) - 2, -1, -1)
        prev = 0
        for i in rng:
            j = i - 1 if p % 2 == 0 else i + 1
            qf[i] = alpha * qf[j] + (1 + alpha) / 2 * (b[i] - b[j])
            qf[i] = min(max(qf[i], 0.0), b[i])
        b = b - qf
    return b


Qclean = pd.Series(Qobs, index=df.index).interpolate().to_numpy()   # fill small gaps
baseflow_obs = lyne_hollick(Qclean)
kperc_grid = np.linspace(0.005, 0.30, 30)


In [ ]:
# ==== YOUR CODE ====
# 1) observed baseflow index over EVAL
BFI_obs = None        # <-- replace: baseflow_obs[EVAL].sum() / Qclean[EVAL].sum()

# 2) model BFI for each kperc in kperc_grid
BFI_model = []
for kp in kperc_grid:
    pars = dict(BASE); pars["kperc"] = kp
    # qf, qs = ... run_split ...
    # BFI_model.append( qs[EVAL].sum() / (qf[EVAL] + qs[EVAL]).sum() )
    BFI_model.append(np.nan)          # <-- replace the two lines above
BFI_model = np.array(BFI_model)

# 3) kperc whose model BFI is closest to the observed BFI
best_kperc = None      # <-- replace: kperc_grid[np.argmin(np.abs(BFI_model - BFI_obs))]
# ===================

print(f"observed BFI (Lyne-Hollick) : {BFI_obs}")
print(f"best-matching kperc         : {best_kperc}")


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

ax[0].plot(kperc_grid, BFI_model, "o-", label="model BFI")
if BFI_obs is not None:
    ax[0].axhline(BFI_obs, color="k", ls="--", label=f"observed BFI = {BFI_obs:.2f}")
if best_kperc is not None:
    ax[0].axvline(best_kperc, color="tab:red", lw=1)
ax[0].set_xlabel("$k_{perc}$"); ax[0].set_ylabel("baseflow index")
ax[0].set_title("tuning one parameter to one signature"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)

sl = slice("2005", "2007")
S = lambda a: pd.Series(a, index=df.index).loc[sl]
ax[1].plot(S(Qclean).index, S(Qclean), color="black", lw=0.8, label="observed Q")
ax[1].fill_between(S(baseflow_obs).index, 0, S(baseflow_obs), color="tab:blue", alpha=0.4,
                   label="filtered baseflow")
ax[1].set_ylabel("Q [mm/d]"); ax[1].set_xlabel("year")
ax[1].set_title("Lyne-Hollick baseflow separation (observed)"); ax[1].legend(fontsize=8)
ax[1].grid(alpha=0.3)
fig.tight_layout()


```{admonition} Answers
:class: note
1. What is the observed BFI, and what $k_{perc}$ matches it? Is the Lecture 7
   hand-picked value ($k_{perc} = 0.05$) close?
2. Does matching the BFI also improve the daily **NSE**? Run the model with
   `best_kperc` and check. Why might a model that matches a signature still fit the
   hydrograph poorly (or vice versa)?
3. The Lyne&ndash;Hollick filter has its own parameter $\alpha$. Re-run the filter with
   $\alpha = 0.9$ and $0.95$. How much does the "observed" BFI &mdash; your calibration
   target &mdash; move? What does that say about calibrating to filtered quantities?
```


## Exercise 3 &mdash; are calibrated parameters climate-transferable?

A model calibrated on today's climate is routinely used to simulate a *future* climate.
That only works if the parameters do not themselves depend on climate. Here you stress-
test that assumption with a **delta-change** experiment &mdash; the same technique used
for climate-impact studies in Module 4.

**Background.** A delta-change scenario perturbs the observed forcing by fixed amounts:
raise every temperature by $\Delta T$, and scale every precipitation value by a factor
$(1 + \Delta P)$. Potential evapotranspiration rises with temperature; a simple rule
consistent with the Oudin formula used for this dataset is

$$E_p^{\text{new}} = E_p \,\bigl(1 + 0.05\,\Delta T\bigr)\qquad(\text{about }5\%\text{ per }°C).$$

You then run the **baseline (already calibrated) parameter set** on the perturbed
forcing and look at how streamflow and the water balance respond.

**Your task.**
1. Complete `perturb(P, Ep, T, dT, dP)` to return perturbed `(P2, Ep2)`.
2. Run the model for a grid of scenarios: $\Delta T \in \{0, 1, 2, 3\}$ °C and
   $\Delta P \in \{-0.10, 0, +0.10\}$.
3. For each, record the change in mean annual streamflow relative to the baseline.
   The given plot draws the response surface.

**Hints.**
* `P2 = P * (1 + dP)` and `T2 = T + dT`; only `P2` and `Ep2` are needed to run the
  model, but compute `Ep2` from `dT`, not from `T2`.
* Mean annual streamflow = `q.mean() * 365.25` (q is in mm/day).
* Use the baseline `s0=0.5` and score over `EVAL` as before; here you care about the
  long-term mean, not NSE.


In [ ]:
def perturb(P, Ep, T, dT, dP):
    """Delta-change perturbation of the forcing.

    dT : temperature change [degC]
    dP : fractional precipitation change (e.g. -0.1 for -10%)

    Returns (P2, Ep2): perturbed precipitation and potential ET, same length as inputs.
    """
    # ==== YOUR CODE ====
    # P2  : scale precipitation by (1 + dP)
    # Ep2 : scale potential ET by (1 + 0.05 * dT)
    P2  = P            # <-- replace
    Ep2 = Ep           # <-- replace
    # ===================
    return P2, Ep2


# --- scenario grid (given, once perturb works) ---
dTs = [0.0, 1.0, 2.0, 3.0]
dPs = [-0.10, 0.0, 0.10]

Q_base_annual = Catchment(**BASE).run(P, Ep)[EVAL].mean() * 365.25
resp = np.zeros((len(dPs), len(dTs)))
for i, dP in enumerate(dPs):
    for j, dT in enumerate(dTs):
        P2, Ep2 = perturb(P, Ep, T, dT, dP)
        q2 = Catchment(**BASE).run(P2, Ep2)
        resp[i, j] = q2[EVAL].mean() * 365.25 - Q_base_annual

print("change in mean annual streamflow [mm/yr] vs baseline:")
print(pd.DataFrame(resp, index=[f"dP={p:+.0%}" for p in dPs],
                   columns=[f"dT={t:+.0f}" for t in dTs]).round(0))


In [ ]:
# --- plot (given) ---
fig, ax = plt.subplots(figsize=(7, 4.5))
im = ax.imshow(resp, origin="lower", aspect="auto", cmap="BrBG",
               vmin=-abs(resp).max(), vmax=abs(resp).max())
ax.set_xticks(range(len(dTs)), [f"+{t:.0f}" for t in dTs])
ax.set_yticks(range(len(dPs)), [f"{p:+.0%}" for p in dPs])
ax.set_xlabel("temperature change [degC]"); ax.set_ylabel("precipitation change")
ax.set_title("streamflow response to a delta-change scenario\n(baseline parameters, held fixed)")
for i in range(len(dPs)):
    for j in range(len(dTs)):
        ax.text(j, i, f"{resp[i, j]:+.0f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, label="mean annual Q change [mm/yr]")
fig.tight_layout()


```{admonition} Answers
:class: note
1. Warming alone (move along the $\Delta P = 0$ row): does streamflow rise or fall, and
   why &mdash; which term in the water balance is responding?
2. Compare the $+2$ °C / $0\%$ cell with the $0$ °C / $-10\%$ cell. Can a temperature
   change masquerade as a rainfall change in this model?
3. This experiment holds the parameters fixed. Name one physical reason a real
   catchment's *effective* parameters (say $S_{max}$, or the ET function) might
   themselves shift in a warmer climate &mdash; which would make this projection wrong.
```


## Where this goes next

* Fitting these parameters properly, instead of using the hand-picked set, is
  **Lecture 8** and its exercise notebook.
* Routing the streamflow you produced here down a channel network is **Lecture 10**.
* The delta-change method of Exercise 3 is treated in full &mdash; with real CMIP6
  data, bias correction and downscaling &mdash; in **Module 4**.

```{note} Sources
Original teaching material for **CE524 Applied Hydroclimatology**; uses the model and
Chattooga data of Lecture 7 (USGS, Daymet, CAMELS). Text under CC BY-SA 4.0; see
[CREDITS.md](https://github.com/drvivekhydro/hydroclimatology/blob/main/CREDITS.md).
```
